# SOP-UQ 快速上手

这是 [英文 Quick Start](quick_start.ipynb) 的中文版。**选择一个版本运行即可**；两个版本共用 `.runs/notebook` 中的结果，不需要各跑一遍。

先按 [README](../README.md) 安装环境、打开 Jupyter，再用**试用 token**从上往下运行。在 Jupyter 中选中一个代码单元，按 **Shift + Enter** 执行，等它完成后继续下一格。你会依次：

1. 连接本队考官和千问模型服务；
2. 查看数据，手动调用业务工具和 oracle；
3. 运行基线，采集有标签题的轨迹并拟合风险估计器；
4. 完成问题 1：估计 360 道测试题的初始风险并提交；
5. 完成问题 2：核验交互题，形成最终决策并提交。

**现在准备下面四项连接信息**，都由助教提供：

| 配置变量 | 填什么 |
| --- | --- |
| `EXAMINER_URL` | 本队考官的完整 URL |
| `EXAMINER_TOKEN` | 本队试用 token |
| `MODEL_URL` | 模型服务的完整 URL |
| `MODEL_TOKEN` | 模型 token |

正式 token 先保存好，后面正式运行时再替换 `EXAMINER_TOKEN`。试用提交只检查格式，不评分；试用 token 在测试题上的工具和 oracle 调用有少量额度。

如果已经按 README 设置环境变量，下方代码会自动读取，无需再填。如果没有设置，可以把下方 `os.environ.get(...)` **第二个参数引号里的内容**改成收到的地址或 token，变量名和第一个参数保持原样。例如 `EXAMINER_TOKEN` 对应行末的空字符串 `""` 可以填入试用 token。`127.0.0.1` 是默认的本机地址，并不是助教提供的服务地址。

第 6、7 节都会提交 CSV。用正式 token 跑这些提交示例会占用后续提交次数。**正式运行时按阶段执行，不要点 Run All**：先完成问题 1 的改进和提交，再进入问题 2。项目介绍与规则见 [考生操作手册](../docs/candidate_manual.pdf)，任务定义见 [题面](../docs/problem_statement.pdf)。


In [ ]:
from pathlib import Path
import json, os, sys

# 从项目根目录或 scripts/ 启动 Jupyter，都能定位项目。
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "test_client").is_dir())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
WORK = ROOT / ".runs" / "notebook"          # 本 Notebook 的所有输出（Git 忽略）

# 考官和模型的连接配置。可在启动 Jupyter 前设置环境变量，
# 或在下方填写；分享 Notebook 前请清除 token。
EXAMINER_URL = os.environ.get("SOP_EXAMINER_URL", "http://127.0.0.1:8801")
EXAMINER_TOKEN = os.environ.get("SOP_EXAMINER_TOKEN", "")
MODEL_URL = os.environ.get("SOP_MODEL_URL", "http://127.0.0.1:18080")
MODEL_TOKEN = os.environ.get("SOP_MODEL_TOKEN", "")

## 1. 连接服务

`spec()` 会显示当前身份：`trial` 表示试用，`exam` 表示正式；还会显示计费规则、每题额度，以及 360 道测试题中哪些允许继续调用工具和 oracle。正式身份下还会显示剩余提交次数，以及问题 1 是否仍可更新。

准备正式运行时，把顶部配置单元中 `EXAMINER_TOKEN` 所在的整行改成下面这样，并填入实际 token：

```python
EXAMINER_TOKEN = "助教发放的正式 token"
```

这样会直接使用你填写的 token。若环境变量里已有试用 token，只修改 `os.environ.get` 的默认值不会覆盖它。然后重新运行顶部配置单元和下面的连接单元，确认输出为 `身份: exam`，并查看剩余提交次数。另开终端执行 `export` 不会改变当前 Jupyter 内核的环境变量。


In [ ]:
from test_client import ExaminerClient, ModelClient, RemoteError

examiner = ExaminerClient(EXAMINER_URL, EXAMINER_TOKEN)
spec = examiner.spec()
print("身份:", spec["mode"], "| 计费:", spec["costs"], "| 每题预算:", spec["budget"],
      "| 工具调用上限:", spec["tool_limit"], "| oracle 次数上限:", spec["oracle_limit"])
print("测试题数:", len(spec["test_ids"]), "| 可交互题数:", len(spec["interactive_ids"]),
      "| 是否冻结初始作答:", spec["frozen_initials"])
if spec["mode"] == "trial":
    print("测试题试用额度:", spec["trial_quota"])
else:
    print("剩余提交次数:", spec["submissions_left"], "/", spec["max_submissions"],
          "| 问题 1 关闭时间:", spec["problem1_closed_at"])

model = ModelClient(MODEL_URL, MODEL_TOKEN)
print("模型:", model.model_id)

## 2. 查看数据

有标签题放在 `data/train/`，分为 `train`、`cal_fit`、`cal_select`、`dev` 四份。测试输入放在 `data/test/cases.jsonl`，对应的冻结初始作答放在 `data/test/initial_M9.jsonl`。冻结作答是模型回答，不是测试题的标准答案。

660 道有标签题已经附带千问的初始作答和轨迹。本 Notebook 演示自行采集的完整流程；若想直接使用现成记录，见 [README 中的可选离线流程](../README.md#可选直接使用老师提供的初始作答)。这些记录只有查询前轨迹；拟合查询后风险所需的轨迹在第 5 节采集。


In [ ]:
from collections import Counter
from baselines.runner import LABELED_SPLITS, labeled_cases, load_cases, load_frozen

labeled = load_cases(DATA)
test_cases = load_cases(DATA, ["test"])
frozen = load_frozen(DATA)
print(Counter((c["training_split"], c["domain"]) for c in labeled))
print("测试题数:", len(test_cases), "| 冻结初始作答数:", len(frozen))

example = {d: next(c for c in labeled if c["domain"] == d and c["training_split"] == "train")
           for d in spec["domains"]}
for domain, case in example.items():
    print(f"\n{domain}: 输入字段 {sorted(case['inputs'])}\n  标准答案 {case['label']}")

## 3. 手动调用工具和 oracle

一次 *episode* 表示打开一道题进行操作。有标签题每次打开都是新的练习；测试题则为每队保留同一份记录，调用和费用持续累计。

普通工具调用费用为 0.01；对有噪声工具使用 `mode="independent"` 时为 0.03。同一道题第 n 次新的 oracle 核验收费 `0.10 × n`，即 0.10、0.20、0.30……。重复查询同一字段、同一阶段免费，也不增加 n。每题预算为 2.00，`view.oracle_pricing` 给出下一次核验的价格。


In [ ]:
def input_tool(episode, name, mode="default"):
    """调用必填参数均可从题目输入中取得的工具。"""
    tool = next(t for t in episode["view"]["tools"] if t["name"] == name)
    required = tool["inputSchema"]["json"].get("required", [])
    args = {k: episode["view"]["inputs"][k] for k in required}
    return examiner.tool(episode["episode_id"], name, args, mode)["receipt"]

dg = examiner.episode(example["dangerous_goods"]["case_id"])
print("可用工具:", [t["name"] for t in dg["view"]["tools"]])
print(input_tool(dg, "calculate_sds_label_score"))
print(input_tool(dg, "calculate_sds_label_score", mode="independent"))   # 第二个数据源
print(examiner.oracle(dg["episode_id"], "sds", "assessment")["receipt"])   # 字段真值
history = examiner.history(dg["episode_id"])
print("当前累计费用:", history["cost"], "| 工具调用次数:", history["tool_calls"],
      "| oracle 调用次数:", history["oracle_calls"])

In [ ]:
air = examiner.episode(example["aircraft_inspection"]["case_id"])
print(input_tool(air, "VerifyAircraftClearance"))

cs = examiner.episode(example["customer_service"]["case_id"])
print(input_tool(cs, "validateAccount"))
try:   # 客服指标在完成诊断后才能查询，因此这里会被拒绝
    examiner.oracle(cs["episode_id"], "latency", "pre_repair")
except RemoteError as error:
    print("查询被拒绝:", error)

## 4. 用千问跑一道基线题

Agent 读取 SOP，逐步调用工具，最后给出 `final_proposal`，其中包含模型自报的出错概率。随后，基于同一份证据再进行四次不调用工具的采样，根据回答之间的差异计算 `disagreement` 风险。


In [ ]:
from baselines.agent import INITIAL_CALL_LIMIT, main_loop, new_snapshot, shared_samples
from baselines.uq import disagreement

case = example["dangerous_goods"]
snapshot = new_snapshot(examiner.episode(case["case_id"]), model)
main_loop(snapshot, model, examiner, limit=INITIAL_CALL_LIMIT)
shared_samples(snapshot, model)
print("模型答案:", snapshot["answer"], "| 标准答案:", case["label"])
print("模型自报风险:", snapshot["p_error"], "| 回答分歧风险:", disagreement(snapshot))
tools = [e["receipt"]["tool"] for e in snapshot["history"] if e["kind"] == "tool_receipt"]
print("调用的工具:", tools, "| 费用:", snapshot["cost"])

## 5. 采集有标签轨迹并拟合风险估计器

完成顶部配置以及第 1、2 节的连接和数据读取后，运行本节的**全部代码单元**，准备正式基线流程所需的 `calibrator`、`METHOD` 和结果读取函数。再次进入这套流程时，可以跳过第 3、4 节的手动演示。

一次采集同时生成查询前和查询后记录：`run_cases` 按 `fixed` 策略运行每道有标签题，得到初始答案和四次采样，再核验一个 oracle 字段，让 Agent 继续作答。`initial.jsonl` 用于拟合初始风险，`posterior.jsonl` 用于拟合两类查询后风险。每一步都会保存进度，相同配置下重新运行会接着已有进度继续。

`LIMIT` 表示每个数据划分取多少题，默认先少量试跑。要完成基线的全量采集，将 `LIMIT` 改为 `None` 后重新运行，补齐到 660 题，再重新拟合。单卡平均每题约 10 秒，客服题约 15 秒，660 题约需 2 小时，可以提前开始。`fit` 使用 `train` 学习历史错误率和监督估计器，使用 `cal_fit` 校准概率，使用 `cal_select` 选择转人工阈值。

下方采集调用单独设置了 `policy="fixed", fixed_count=1`。只修改后面的 `METHOD` 不会改变这里的采集设置。如果改变测试时的查询策略，也要用对应策略采集有标签题的查询后轨迹，并重新拟合查询后风险。


In [ ]:
from baselines.runner import run_cases
from baselines.storage import read_jsonl
from baselines.uq import Calibrator

LIMIT = 4   # 首次每个划分先取少量题；None 表示全部 660 题
cases = labeled_cases(DATA, LABELED_SPLITS, LIMIT)
print(run_cases(examiner, cases, WORK / "collect", model=model, policy="fixed", fixed_count=1))

initial = read_jsonl(WORK / "collect" / "initial.jsonl")
posterior = read_jsonl(WORK / "collect" / "posterior.jsonl")
calibrator = Calibrator().fit(labeled, initial)
calibrator.select_thresholds(labeled, initial, split="cal_select")
calibrator.fit_posterior(labeled, posterior)
try:
    calibrator.select_thresholds(labeled, posterior, split="cal_select", posterior=True)
except ValueError:
    print("目前还没有 cal_select 的查询后记录；查询后暂用初始风险的阈值。")
calibrator.save(WORK / "calibrator.json")
print(calibrator.fit_counts)
print({method: round(record["threshold"], 3) for method, record in calibrator.selections.items()})

### 两类查询后风险

读取 oracle 后，Agent 会继续作答，给出两个风险：

- `p_initial_error_after_queries`：知道新信息后，重新判断**初始答案**出错的概率，即 q；
- `p_final_error`：判断**最终答案**出错的概率，即 r。

上面的 `fit_posterior` 已用有标签题的查询后轨迹校准这两个风险。下面的 `METHOD` 集中设置**后续测试运行**使用的方法；前面的采集调用仍单独设置查询策略，两处应保持一致。


In [ ]:
METHOD = dict(uq="calibrated_disagreement", policy="fixed", fixed_count=1)

### 在本地给有标签结果评分

每次运行都会生成 `results.csv`。有标签题可以在本地按考官使用的规则计算细分指标，不占提交次数。

输出中的 `initial` 是查询前初始风险，`updated_initial` 是查询后的初始风险，`final` 是最终答案风险，`decision` 是决策指标。


In [ ]:
from test_client import read_results, score
from test_client.results import check_results

def local_score(path):
    index = {c["case_id"]: c for c in labeled}
    rows = check_results(read_results(path), index, set(index), require_all=False)
    return score(rows, {cid: index[cid] for cid in rows}, set(index))

report = local_score(WORK / "collect" / "results.csv")
for section in ("initial", "updated_initial", "final"):
    print(section, report[section]["domain_macro"])
print("决策指标", report["decision"]["domain_macro"])

## 6. 问题 1：估计测试题风险并提交

考官为每道测试题固定了一份初始作答，各队拿到的相同。本节不读取 oracle、不调用模型：估计全部 360 题的 `p_initial_error`，并仅根据这项风险，为 72 道交互题选择 `AUTO`（自动采用）或 `DEFER`（转人工）。

**下面的代码单元会生成并提交 CSV。** 使用正式 token 时，每次有效上传占一次提交并立即评分；格式不通过的文件会被拒绝，重复上传完全相同的文件会返回原报告，这两种情况都不占次数。试用 token 只检查格式。评分反馈用于评价自己的方法，不用于推断测试标签。在终端中也可以通过 `scripts/run_test.py` 提交。

切换正式 token 并重新运行第 1、2、5 节后，先执行本节，再进入第 7 节。本节会根据当前身份设置 `RUN`：试用结果存入 `WORK / "trial"`，正式结果存入 `WORK / "exam"`。尝试新方法时，如需保存一套新的运行文件，可以在这里改成 `RUN = WORK / spec["mode"] / "v2"`；第 7、8 节会沿用这个 `RUN`。


In [ ]:
RUN = WORK / spec["mode"]                  # 试用和正式测试结果分别保存
problem1 = run_cases(examiner, test_cases, RUN / "problem1", model=model, frozen=frozen,
                     calibrator=calibrator, **dict(METHOD, policy="threshold"))
print(problem1)

def show(answer):
    if answer.get("trial"):
        print("试用身份：文件格式通过；不评分，不占提交次数")
        return
    print("提交编号", answer["submission"], "| 剩余次数:", answer["submissions_left"],
          "| 原始细分指标最佳记录:", answer["best"])
    for section in ("initial", "updated_initial", "final", "decision"):
        print(section, answer["report"][section]["domain_macro"])

results = RUN / "problem1" / "results.csv"
test_index = {c["case_id"]: c for c in test_cases}
check_results(read_results(results), test_index, set(spec["interactive_ids"]))
show(examiner.submit(results))             # 正式 token：占一次提交

## 7. 问题 2：核验、决策并提交

现在，运行器会为每道交互题核验一个 oracle 字段，让 Agent 继续作答，估计 q/r，选择 `AUTO` 或 `DEFER`，再提交 CSV。查询测试题 oracle 前，须先提交一次结果。首次成功查询正式测试 oracle 后，**p 不再更新，只更新 q/r 和决策成绩**。

完成问题 1 的改进和提交后，再将 `DONE_WITH_PROBLEM_1` 设为 `True`。如果要用第 8 节的示例改进 p，应先运行并提交那个示例，再执行本节。使用试用 token 时，无需修改这个开关，本节会在试用额度内运行。

原报告中的 `best.problem2` 记录决策代价最低的那次提交。`report.grades` 将本次 q/r 和决策成绩与 oracle 前可计入的最佳 p 成绩组合，得到本次有效成绩并同步到大屏：详情榜显示最新一次，首页排名保留历史最高总分。`report.submission_grades` 保留这份 CSV 独立计算的成绩，其中 p 没有应用历史择优，供分析使用。完整评分规则见 [考生操作手册](../docs/candidate_manual.pdf)。


In [ ]:
DONE_WITH_PROBLEM_1 = False   # 正式 token：完成问题 1 的提交后改为 True

if spec["mode"] == "trial" or DONE_WITH_PROBLEM_1:
    problem2 = run_cases(examiner, test_cases, RUN / "problem2", model=model, frozen=frozen,
                         calibrator=calibrator, **METHOD)
    print(problem2)
    show(examiner.submit(RUN / "problem2" / "results.csv"))
else:
    print("已跳过：问题 1 仍可更新；准备进入问题 2 时，将 DONE_WITH_PROBLEM_1 设为 True。")
listing = examiner.submissions()
for item in listing["submissions"]:
    print("提交编号", item["submission"], "| p_i 的 Brier:", item["primary"])
print("原始细分指标最佳记录:", listing["best"], "| 问题 1 关闭时间:", listing["problem1_closed_at"])

## 8. 尝试自己的方法

可以自行设计方法，只需输出同样格式的 `results.csv`：每道测试题一行，包含 `p_initial_error`；交互题还需要填写 `mode`、最终答案、`p_initial_error_after_queries` 和 `p_final_error`。初始答案必须使用冻结作答，p 只在问题 1 关闭前的提交中计入更新。

下面演示一种简单修改：把问题 1 的风险估计向该业务的历史错误率适当调整，再检查结果文件。

**下面的单元只生成并检查 `my_results.csv`，不会自动提交。** 要采用这组 p，请在第 6 节之后运行，并在第 7 节之前提交。在项目根目录打开已配置好目标考官地址和 token 的终端，运行 `python scripts/run_test.py`，后面接下方打印的文件路径。使用默认正式运行目录时，命令为：

```bash
python scripts/run_test.py .runs/notebook/exam/my_results.csv
```


In [ ]:
from test_client import write_results

rows = read_results(RUN / "problem1" / "results.csv")
domain_of = {c["case_id"]: c["domain"] for c in test_cases}
for case_id, row in rows.items():
    prior = calibrator.rates[domain_of[case_id]]
    if row["p_initial_error"] is not None:
        row["p_initial_error"] = 0.7 * row["p_initial_error"] + 0.3 * prior
mine = write_results(RUN / "my_results.csv", [{"case_id": k, **v} for k, v in rows.items()])
check_results(read_results(mine), test_index, set(spec["interactive_ids"]))
print("已生成可提交文件:", mine)